In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Dataset Generation — Handwriting Patches

Pipeline for generating the `.zarr` patch datasets used in all training experiments.

**Input:** raw handwriting images (one per author) + `.mat` metadata files with line segmentation coordinates.
**Output:** one `.zarr` archive per image variant, compressed as `.tar.gz` and saved to Google Drive.

**Authors:** 407 total, sorted by natural key (`lines1_Page_01`, `lines1_Page_02`, ..., `lines7_Page_62`).

**Image variants processed:**
| Folder | Description |
|---|---|
| `1_ImagesRotated` | Deskewed original |
| `2_ImagesMedianBW` | Median-filtered binary |
| `3_ImagesLinesRemovedBW` | Lines removed, binary |
| `4_ImagesLinesRemoved` | Lines removed, grayscale |

In [ ]:
import re
import os
import glob
from tqdm import tqdm
import datetime
import gc
import numpy as np
import shutil
import random
import tensorflow as tf
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable

In [ ]:
!pip install "zarr>=2.16.0,<3.0.0" numcodecs visualkeras pillow

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 154.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.6 MB/s eta 0:00:00
  Created wheel for asciitree: filename=asciitree-0.3.3-py3-none-any.whl size=5031 sha256=b0e02ec5ed571a5eb1c333b961ef9c3839fa92eff407206a705c65527d7d5a36
  Stored in directory: /root/.cache/pip/wheels/a5/d7/98/f56ae733748cd0fa577172bda0e73e0b1f1793c98e09b9e458
Successfully built asciitree


In [ ]:
!pip install keras_hub keras

## Configuration

- `DataSourceConfig` — paths to raw images and `.mat` metadata files, plus per-folder empty patch thresholds
- `PatchConfig` — patch size and stride for the sliding window extractor

**Patch extraction parameters:**
- `patch_size=(180, 360)` — height × width in pixels
- `strides=(30, 90)` — vertical and horizontal stride of the sliding window

**Empty patch thresholds** — patches where the fraction of white pixels exceeds the threshold are discarded:
- `1_ImagesRotated`, `2_ImagesMedianBW` → `0.95`
- `3_ImagesLinesRemovedBW`, `4_ImagesLinesRemoved` → `0.97`

Higher thresholds for line-removed variants because background is cleaner and more uniform.

In [ ]:
@dataclass
class DataSourceConfig:
  base_path: Path
  folders: list[str]
  mat_path: Path
  thresholds: dict[str, float]

  def __post_init__(self):
        # Validate folders vs thresholds
        missing = set(self.folders) - set(self.thresholds.keys())
        if missing:
            raise ValueError(f"Missing thresholds for folders: {missing}")


@dataclass
class PatchConfig:
  patch_size: tuple[int, int]
  strides: tuple[int, int]


@dataclass
class Config:
    project_path: Path
    seed: int
    current_time: str
    batch_size: int
    data_source: DataSourceConfig
    patch: PatchConfig

In [ ]:
config = Config(
    project_path=Path('/content/drive/MyDrive/author-handwriting-recog'),
    seed=42,
    current_time=datetime.datetime.now().strftime("%Y%m%d-%H%M%S"),
    batch_size=256,
    data_source=DataSourceConfig(
        base_path=Path('/content/drive/MyDrive/handwriting data/'),
        folders=['1_ImagesRotated','2_ImagesMedianBW', '3_ImagesLinesRemovedBW', '4_ImagesLinesRemoved'],
        mat_path=Path('/content/drive/MyDrive/handwriting data/5_DataDarkLines'),
        thresholds={
          '1_ImagesRotated': 0.95,
          '2_ImagesMedianBW': 0.95,
          '3_ImagesLinesRemovedBW': 0.97,
          '4_ImagesLinesRemoved': 0.97
        },
    ),
    patch=PatchConfig(
        patch_size=(180, 360),
        strides=(30, 90)
    )
)

In [ ]:
def upload_source(source: str, source_dest: str):
    if os.path.exists(source_dest):
        shutil.rmtree(source_dest)

    shutil.copytree(source, source_dest)

In [ ]:
upload_source(source=f"{config.project_path}/src", source_dest="./src")

In [ ]:
random.seed(config.seed)
np.random.seed(config.seed)
tf.random.set_seed(config.seed)

In [ ]:
from src.io import (
    LoggerFactory,
    ImageTransformer,
    ImageAnalyzer,
    save,
    FileSystem,
    load,
    PatchExtractor
)

In [ ]:
from src.datasets import (
    AuthorMetadataLoader,
    AuthorDatasetBuilder,
    AuthorInfo,
    AuthorDataset,
    PatchDataset,
)

In [ ]:
fileSystem = FileSystem()

## Author Metadata

Each `.mat` file contains:
- **Line coordinates** — vertical boundaries of each handwritten line
- **Test area** — a designated spatial region (top/bottom pixel rows) reserved exclusively for testing
- **Scale factor** and **line height** — used to normalize coordinates across images

Author names are sorted using `natural_key` (numeric-aware sort) before label assignment.
This ensures **deterministic and consistent author IDs across all dataset runs**.

> The test area is physically separated from the training area in the original image.

> No patch can span both regions — this is enforced at extraction time.

In [ ]:
def natural_key(name: str) -> list:
  return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', name)]

In [ ]:
# List of all the files names
authors: list = glob.glob(f"{config.data_source.mat_path}/*.mat")

# get only the file name without the extension
authors_names: list = [os.path.splitext(os.path.basename(author))[0]
                for author in authors]

authors_names: list[str] = sorted(authors_names, key=natural_key)

In [ ]:
author_metadata_loader = AuthorMetadataLoader(
    parent_path=config.data_source.mat_path,
    load_fn=load
)

In [ ]:
logger = LoggerFactory.get_logger(
    "dataset_generation",
    log_dir=f"./logs_{config.current_time}",
    file_prefix=f"_{config.current_time}"
)

In [ ]:
def build_authors_info(
    authors: list[str],
    author_load: AuthorMetadataLoader
) -> dict[str, AuthorInfo]:
    pbar = tqdm(authors, desc="Processing authors")
    author_infos = {}

    for author in pbar:
        pbar.set_postfix(current_author=author)
        info = author_load.build_author_data(author)
        author_infos[author] = info

    return author_infos

In [ ]:
author_infos = build_authors_info(authors_names, author_metadata_loader)

Processing authors: 100%|██████████| 407/407 [00:09<00:00, 42.01it/s, current_author=lines7_Page_62] 


In [ ]:
author_infos['lines1_Page_01']

AuthorInfo(author_id='lines1_Page_01', lines=[(1415, 1570), (1570, 1735), (1735, 1895), (1895, 2060), (2060, 2225), (2225, 2385), (2385, 2550), (2550, 2715), (2715, 2880), (2880, 3045), (3045, 3205), (3205, 3365), (3365, 3525), (3525, 3690), (3690, 3855), (3855, 4015), (4015, 4180), (4180, 4345), (4345, 4505), (4505, 4670), (4670, 4835), (4835, 4995), (4995, 5160), (5160, 5325), (5325, 5490), (5490, 5655), (5655, 5820), (5820, 5980), (5980, 6140)], test_area_top=4038, test_area_bottom=4219, scale_factor=5.0, line_height=181)

## Dataset Generation

For each image variant, the pipeline:

1. Iterates over all 407 authors in sorted order
2. Loads the author image from the corresponding folder
3. Extracts patches using a sliding window, filtered by the empty threshold
4. Separates patches into `test` (from test area) and `non_test` (from the rest)
5. Saves patches and labels incrementally to the `.zarr` archive

After all authors are processed, metadata is saved to the zarr root attributes:

```
zarr.attrs = {
    "author_names": [...],   # ordered list of all author names
    "num_authors": 407,
    "patch_shape": (180, 360),
    "strides": (30, 90),
    "threshold": 0.97,
    "seed": 42,
    "created_at": "YYYYMMDD-HHMMSS"
}
```

> `author_names` is critical — it maps integer label IDs back to author names

> and is used downstream by `SelectiveDataLoader` and `EmbeddingVisualizer`.

In [ ]:
def normalize_data(data: AuthorDataset):
    return {
            "test": {
                "images": data.test.images,
                "labels": data.test.labels,
            },
            "non_test": {
                "images": data.non_test.images,
                "labels": data.non_test.labels,
            },
        }

In [ ]:
def generate_dataset(
    author_infos: dict[str, AuthorInfo],
    dataset_builder: AuthorDatasetBuilder,
    save_fn: Callable[[str, dict], None],
    directory: str,
    filename: str,
) -> list[dict[str, str]]:
    failed_authors = []
    pbar = tqdm(
        author_infos.keys(),
        desc="Processing authors",
    )

    for label_id, author in enumerate(pbar):
        pbar.set_postfix(current_author=author)
        try:
            info = author_infos[author]
            dataset = dataset_builder.build(info=info, label_id=label_id)
            dataset = normalize_data(dataset)
            save_fn(directory, data=dataset, filename=filename)
        except ValueError as e:
          logger.warning(f"Skipping author {author}: {e}")
          failed_authors.append((author, str(e)))
        except Exception as e:
          logger.error(f"Unexpected error processing author {author}: {e}", exc_info=True)
          failed_authors.append((author, f"CRITICAL: {e}"))

    return failed_authors

In [ ]:
def display_failed(failed_authors: list[dict[str, str]], folder: str) -> None:
    if failed_authors:
        logger.warning(f"Failed to process {folder} - {len(failed_authors)} authors:")
        for author, reason in failed_authors:
            logger.warning(f"  - {author}: {reason}")

In [ ]:
transformer = ImageTransformer()
analyzer = ImageAnalyzer()

In [ ]:
for folder in config.data_source.folders:
    logger.info(f"\nStarting process for {folder}")
    directory="./"
    filename=f"dataset_{folder}_{config.current_time}.zarr"

    patch_extractor = PatchExtractor(
        image_analyzer=analyzer,
        transformer=transformer,
        output_patch_dim=config.patch.patch_size,
        stride_dim=config.patch.strides,
        empty_threshold=config.data_source.thresholds[folder],
    )

    author_dataset_builder = AuthorDatasetBuilder(
        images_dir=config.data_source.base_path / folder,
        transformer=transformer,
        patch_extractor=patch_extractor,
        load_fn=load,
    )

    failed_authors = generate_dataset(
        author_infos=author_infos,
        dataset_builder=author_dataset_builder,
        save_fn=save,
        directory=directory,
        filename=filename,
    )
    display_failed(failed_authors, folder)

    metadata = {
        'author_names': authors_names,
        'num_authors': len(authors_names),
        'patch_shape': config.patch.patch_size,
        'strides': config.patch.strides,
        'threshold': config.data_source.thresholds[folder],
        'seed': config.seed,
        'created_at': config.current_time,
    }

    save(
        directory=directory,
        data={"metadata": metadata},
        filename=filename,
    )


18:08:27 - dataset_generation - INFO - 
Starting process for 1_ImagesRotated


Processing authors:   0%|          | 0/407 [00:00<?, ?it/s, current_author=lines1_Page_01]/usr/local/lib/python3.12/dist-packages/zarr/creation.py:190: UserWarning: ignoring keyword argument 'maxshape'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
Processing authors: 100%|██████████| 407/407 [18:07<00:00,  2.67s/it, current_author=lines7_Page_62]

18:26:35 - dataset_generation - INFO - 
Starting process for 2_ImagesMedianBW



Processing authors: 100%|██████████| 407/407 [17:54<00:00,  2.64s/it, current_author=lines7_Page_62]

18:44:29 - dataset_generation - INFO - 
Starting process for 3_ImagesLinesRemovedBW



Processing authors: 100%|██████████| 407/407 [17:38<00:00,  2.60s/it, current_author=lines7_Page_62]

19:02:08 - dataset_generation - INFO - 
Starting process for 4_ImagesLinesRemoved



Processing authors: 100%|██████████| 407/407 [17:32<00:00,  2.59s/it, current_author=lines7_Page_62]


## Export

Each `.zarr` directory is compressed into a `.tar.gz` archive and moved to Google Drive
under `data/datasets/180_360_90/`.

One archive per variant:
- `dataset_1_ImagesRotated_{{TIMESTAMP}}.tar.gz`
- `dataset_2_ImagesMedianBW_{{TIMESTAMP}}.tar.gz`
- `dataset_3_ImagesLinesRemovedBW_{{TIMESTAMP}}.tar.gz`
- `dataset_4_ImagesLinesRemoved_{{TIMESTAMP}}.tar.gz`

> Compression is necessary — raw `.zarr` directories for 407 authors are several GB each.

> The `.tar.gz` archives are the inputs consumed by all training notebooks.

In [ ]:
!tar -czf dataset_1_ImagesRotated_{config.current_time}.tar.gz dataset_1_ImagesRotated_{config.current_time}.zarr

In [ ]:
!tar -czf dataset_2_ImagesMedianBW_{config.current_time}.tar.gz dataset_2_ImagesMedianBW_{config.current_time}.zarr

In [ ]:
!tar -czf dataset_3_ImagesLinesRemovedBW_{config.current_time}.tar.gz dataset_3_ImagesLinesRemovedBW_{config.current_time}.zarr

In [ ]:
!tar -czf dataset_4_ImagesLinesRemoved_{config.current_time}.tar.gz dataset_4_ImagesLinesRemoved_{config.current_time}.zarr

In [ ]:
from pathlib import Path

def save_in_drive(source_dir: Path, destination_dir: Path) -> None:
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / source_dir.name
    shutil.move(source_dir, destination)
    logger.info("✅ Dataset saved successfully!")
    logger.info(f"Moved to: {destination}")

In [ ]:
destination_dir = Path(f"{config.project_path}/data/datasets/180_360_90")

In [ ]:
sources = [
    Path(f"./dataset_1_ImagesRotated_{config.current_time}.tar.gz"),
    Path(f"./dataset_2_ImagesMedianBW_{config.current_time}.tar.gz"),
    Path(f"./dataset_3_ImagesLinesRemovedBW_{config.current_time}.tar.gz"),
    Path(f"./dataset_4_ImagesLinesRemoved_{config.current_time}.tar.gz"),
]

In [ ]:
for source in sources:
    save_in_drive(source, destination_dir)

19:22:48 - dataset_generation - INFO - ✅ Dataset saved successfully!
19:22:48 - dataset_generation - INFO - Moved to: /content/drive/MyDrive/author-handwriting-recog/data/datasets/180_360_90/dataset_1_ImagesRotated_20260330-180341.tar.gz
19:22:49 - dataset_generation - INFO - ✅ Dataset saved successfully!
19:22:49 - dataset_generation - INFO - Moved to: /content/drive/MyDrive/author-handwriting-recog/data/datasets/180_360_90/dataset_2_ImagesMedianBW_20260330-180341.tar.gz
19:22:49 - dataset_generation - INFO - ✅ Dataset saved successfully!
19:22:49 - dataset_generation - INFO - Moved to: /content/drive/MyDrive/author-handwriting-recog/data/datasets/180_360_90/dataset_3_ImagesLinesRemovedBW_20260330-180341.tar.gz
19:22:50 - dataset_generation - INFO - ✅ Dataset saved successfully!
19:22:50 - dataset_generation - INFO - Moved to: /content/drive/MyDrive/author-handwriting-recog/data/datasets/180_360_90/dataset_4_ImagesLinesRemoved_20260330-180341.tar.gz


In [ ]:
datasets_zarr = [
    f"./dataset_1_ImagesRotated_{config.current_time}.zarr",
    f"./dataset_2_ImagesMedianBW_{config.current_time}.zarr",
    f"./dataset_3_ImagesLinesRemovedBW_{config.current_time}.zarr",
    f"./dataset_4_ImagesLinesRemoved_{config.current_time}.zarr",
]

In [ ]:
def get_size_mb(dataset_path: str) -> float:
  from pathlib import Path
  path = Path(dataset_path)

  if not path.is_dir():
    return path.stat().st_size / 1024 / 1024

  total_size = 0
  for file_path in path.rglob("*"):
    if file_path.is_file():
      total_size += file_path.stat().st_size

  return total_size / 1024 / 1024

In [ ]:
def analysis_by_group(num_authors: int, labels: np.ndarray, group: str) -> None:
    logger.info(f"{group} analysis:")
    logger.info(f"{group} by author:")

    per_author = len(labels) / num_authors
    logger.info(f"  Average: {per_author:.0f} patches/autor")

    unique, counts = np.unique(labels, return_counts=True)
    counts_dict = dict(zip(unique, counts))

    if len(counts_dict) > 0:
        logger.info(f"  Min: {min(counts_dict.values())} patches")
        logger.info(f"  Max: {max(counts_dict.values())} patches")
        logger.info(f"  Std: {np.std(list(counts_dict.values())):.1f} patches")

    logger.info(
        f"Is it sufficient? {'✅ Yes' if per_author >= 100 else '⚠️ It could be insufficient'}"
    )

## Dataset Analysis

Verification of the generated datasets before use in training.

**Checks performed per variant:**
- Total patches (test + non_test)
- Average, min, max and std of patches per author
- Test / non-test ratio
- File size in MB and GB

**Sufficiency threshold:** ≥ 100 patches per author in non-test is considered sufficient for stable triplet training. Authors below this threshold are flagged with a warning.

> Patch count imbalance across authors is expected — some handwriting pages have more

> usable content than others. The generators handle this at sampling time.

In [ ]:
import zarr

for dataset_path in datasets_zarr:
    root = zarr.open(dataset_path, mode="r")
    dataset_metadata = dict(root.attrs)
    test_images = root["test"]["images"]
    test_labels = root["test"]["labels"][:]

    non_test_images = root["non_test"]["images"]
    non_test_labels = root["non_test"]["labels"][:]

    dataset_file = fileSystem.to_path(dataset_path)
    total_test = len(test_labels)
    total_non_test = len(non_test_labels)
    total_patches = total_test + total_non_test
    file_size_mb = get_size_mb(dataset_path=dataset_file)
    file_size_gb = file_size_mb / 1024

    logger.info(f"    File: {dataset_file}")
    logger.info(f"    Size: {file_size_mb:,.2f} MB")
    logger.info(f"    Size: {file_size_gb:,.2f} GB")
    logger.info(f"    Authors: {dataset_metadata['num_authors']}")
    logger.info(f"    Total patches: {total_patches:,}")
    logger.info(f"    Test: {total_test:,}")
    logger.info(f"    Non-test: {total_non_test:,}")
    logger.info(f"    Patch shape: {dataset_metadata['patch_shape']}")

    num_authors = dataset_metadata["num_authors"]
    analysis_by_group(num_authors, test_labels, "Test Set")
    logger.info(f"{'-'*60}")
    analysis_by_group(num_authors, non_test_labels, "Non Test Set")
    logger.info(f"{'-'*60}")

    logger.info("📊 Test / Non-Test ratio analysis:")

    total_test = len(test_labels)
    total_non_test = len(non_test_labels)
    total = total_test + total_non_test

    test_ratio = total_test / total
    non_test_ratio = total_non_test / total

    logger.info(f"  Total patches: {total:,}")
    logger.info(f"  Test: {test_ratio*100:.2f}%")
    logger.info(f"  Non-test: {non_test_ratio*100:.2f}%")

    logger.info(f"{'='*60}")

19:34:55 - dataset_generation - INFO -     File: dataset_1_ImagesRotated_20260330-180341.zarr
19:34:55 - dataset_generation - INFO -     Size: 236.61 MB
19:34:55 - dataset_generation - INFO -     Size: 0.23 GB
19:34:55 - dataset_generation - INFO -     Authors: 407
19:34:55 - dataset_generation - INFO -     Total patches: 284,460
19:34:55 - dataset_generation - INFO -     Test: 17,852
19:34:55 - dataset_generation - INFO -     Non-test: 266,608
19:34:55 - dataset_generation - INFO -     Patch shape: [180, 360]
19:34:55 - dataset_generation - INFO - Test Set analysis:
19:34:55 - dataset_generation - INFO - Test Set by author:
19:34:55 - dataset_generation - INFO -   Average: 44 patches/autor
19:34:55 - dataset_generation - INFO -   Min: 28 patches
19:34:55 - dataset_generation - INFO -   Max: 49 patches
19:34:55 - dataset_generation - INFO -   Std: 2.1 patches
19:34:55 - dataset_generation - INFO - Is it sufficient? ⚠️ It could be insufficient
19:34:55 - dataset_generation - INFO - ----